In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter
import gradio as gr
import re # For cleaning strings

# ========== 0. CONFIGURATION & SEED ==========
SEED = 42
EXCLUDE_INDY_500 = True # 是否排除 Indianapolis 500 比賽
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

os.makedirs('./data', exist_ok=True)

# ========== 1. HELPER FUNCTIONS ==========
def clean_string(text):
    if isinstance(text, str):
        text = re.sub(r'\s+', ' ', text) # 替換多個空格為單個空格
        return text.strip()
    return text

def time_to_seconds(time_str):
    if pd.isna(time_str) or not isinstance(time_str, str) or not time_str:
        return np.nan
    
    try:
        if re.match(r'^\d+:\d{2}\.\d+$', time_str): # M:SS.mmm or H:MM:SS.mmm (if H is part of M)
            parts = time_str.split(':')
            if len(parts) == 2: # M:SS.mmm
                m, s_ms = parts
                s, ms = s_ms.split('.')
                return int(m) * 60 + int(s) + int(ms) / 1000.0
            elif len(parts) == 3: # H:MM:SS.mmm
                 h, m, s_ms = parts
                 s, ms = s_ms.split('.')
                 return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0
        elif re.match(r'^\d+\.\d+$', time_str): # S.mmm
            s, ms = time_str.split('.')
            return int(s) + int(ms) / 1000.0
    except ValueError:
        return np.nan
    return np.nan 

GP_COUNTRY_MAP = {
    'British': 'GBR', 'Great Britain': 'GBR',
    'Monaco': 'MON',
    'Italian': 'ITA', 'Italy': 'ITA',
    'German': 'GER', 'Germany': 'GER',
    'Belgian': 'BEL', 'Belgium': 'BEL',
    'French': 'FRA', 'France': 'FRA',
    'Dutch': 'NED',
    'Spanish': 'ESP', 'Spain': 'ESP',
    'Brazilian': 'BRA', 'Brazil': 'BRA',
    'Japanese': 'JPN', 'Japan': 'JPN',
    'Canadian': 'CAN', 'Canada': 'CAN',
    'Austrian': 'AUT', 'Austria': 'AUT',
    'Hungarian': 'HUN', 'Hungary': 'HUN',
    'Mexican': 'MEX', 'Mexico': 'MEX',
    'Australian': 'AUS', 'Australia': 'AUS',
    'United States': 'USA', 'USA': 'USA',
    'Swiss': 'SUI', 'Switzerland': 'SUI',
}

def get_country_from_gp(gp_name):
    if not isinstance(gp_name, str): return None
    for key, country_code in GP_COUNTRY_MAP.items():
        if key in gp_name:
            return country_code
    return None

# ========== 2. DATA LOADING AND INITIAL CLEANING ==========
def load_and_clean_data():
    print("Loading and cleaning data...")
    try:
        winners_df = pd.read_csv('./data/winners.csv', encoding='utf-8')
        drivers_df = pd.read_csv('./data/drivers_updated.csv', encoding='utf-8')
        teams_df = pd.read_csv('./data/teams_updated.csv', encoding='utf-8')
        fastest_laps_df = pd.read_csv('./data/fastest_laps_updated.csv', encoding='utf-8')
    except FileNotFoundError as e:
        print(f"Error: Missing CSV file. {e}")
        return None, None, None, None

    for df, cols_to_clean in [
        (winners_df, ['Grand Prix', 'Winner', 'Car']),
        (drivers_df, ['Driver', 'Nationality', 'Car']),
        (teams_df, ['Team']),
        (fastest_laps_df, ['Grand Prix', 'Driver', 'Car'])
    ]:
        if df is None: continue
        for col in cols_to_clean:
            if col in df.columns:
                df[col] = df[col].apply(clean_string)

    if winners_df is not None and 'Date' in winners_df.columns:
        winners_df['year'] = pd.to_datetime(winners_df['Date'], errors='coerce').dt.year
        winners_df.dropna(subset=['year'], inplace=True)
        winners_df['year'] = winners_df['year'].astype(int)
    elif winners_df is not None: 
        raise ValueError("winners.csv is missing 'Date' column.")

    for df, name in [(drivers_df, 'drivers_updated.csv'), (teams_df, 'teams_updated.csv'), (fastest_laps_df, 'fastest_laps_updated.csv')]:
        if df is None: continue
        if 'year' in df.columns:
            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            df.dropna(subset=['year'], inplace=True)
            df['year'] = df['year'].astype(int)
        else:
            raise ValueError(f"{name} is missing 'year' column.")
        
        if 'Pos' in df.columns: 
            df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
            
    if drivers_df is not None and 'Car' in drivers_df.columns:
        drivers_df.rename(columns={'Car': 'Team'}, inplace=True)
    elif drivers_df is not None: 
        raise ValueError("drivers_updated.csv is missing 'Car' column (expected as 'Team').")

    if EXCLUDE_INDY_500:
        print("Excluding Indianapolis 500 races...")
        if winners_df is not None:
            winners_df = winners_df[~winners_df['Grand Prix'].str.contains("Indianapolis 500", na=False)]
        if fastest_laps_df is not None:
            fastest_laps_df = fastest_laps_df[~fastest_laps_df['Grand Prix'].str.contains("Indianapolis 500", na=False)]

    print("Data loading and initial cleaning complete.")
    return winners_df, drivers_df, teams_df, fastest_laps_df

# ========== 3. FEATURE ENGINEERING ==========
def engineer_features(winners_df, drivers_df, teams_df, fastest_laps_df):
    print("Engineering features...")
    if drivers_df is None or teams_df is None or fastest_laps_df is None:
        print("Warning: One or more raw dataframes are None in engineer_features. Will proceed with available data.")
        if drivers_df is None: return None 

    if drivers_df is not None and not drivers_df.empty:
        drivers_df = drivers_df.sort_values(by=['Driver', 'year'])
        if 'PTS' in drivers_df.columns:
            drivers_df['Prev_Year_Driver_PTS'] = drivers_df.groupby('Driver')['PTS'].shift(1).fillna(0)
        else:
            drivers_df['Prev_Year_Driver_PTS'] = 0
            
        max_pos_driver = drivers_df['Pos'].max(skipna=True) if 'Pos' in drivers_df.columns and not drivers_df['Pos'].dropna().empty else 0
        default_prev_pos_driver = (int(max_pos_driver) + 5) if pd.notna(max_pos_driver) and max_pos_driver > 0 else 50 
        if 'Pos' in drivers_df.columns:
            drivers_df['Prev_Year_Driver_Pos'] = drivers_df.groupby('Driver')['Pos'].shift(1).fillna(default_prev_pos_driver)
        else:
            drivers_df['Prev_Year_Driver_Pos'] = default_prev_pos_driver
        
        if 'year' in drivers_df.columns and 'Driver' in drivers_df.columns:
            driver_first_year = drivers_df.groupby('Driver')['year'].min().rename('driver_first_year')
            drivers_df = drivers_df.merge(driver_first_year, on='Driver', how='left')
            if 'driver_first_year' in drivers_df.columns:
                 drivers_df['Driver_Experience_Years'] = drivers_df['year'] - drivers_df['driver_first_year']
            else:
                 drivers_df['Driver_Experience_Years'] = 0
        else:
            drivers_df['Driver_Experience_Years'] = 0
    else: 
        placeholder_cols = ['Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years']
        if drivers_df is not None: 
            for col in placeholder_cols: drivers_df[col] = 0

    if teams_df is not None and not teams_df.empty:
        teams_df = teams_df.sort_values(by=['Team', 'year'])
        if 'PTS' in teams_df.columns:
            teams_df['Prev_Year_Team_PTS'] = teams_df.groupby('Team')['PTS'].shift(1).fillna(0)
        else:
            teams_df['Prev_Year_Team_PTS'] = 0

        max_pos_team = teams_df['Pos'].max(skipna=True) if 'Pos' in teams_df.columns and not teams_df['Pos'].dropna().empty else 0
        default_prev_pos_team = (int(max_pos_team) + 5) if pd.notna(max_pos_team) and max_pos_team > 0 else 20
        if 'Pos' in teams_df.columns:
            teams_df['Prev_Year_Team_Pos'] = teams_df.groupby('Team')['Pos'].shift(1).fillna(default_prev_pos_team)
        else:
            teams_df['Prev_Year_Team_Pos'] = default_prev_pos_team
        
        if 'year' in teams_df.columns and 'Team' in teams_df.columns:
            team_first_year = teams_df.groupby('Team')['year'].min().rename('team_first_year')
            teams_df = teams_df.merge(team_first_year, on='Team', how='left')
            if 'team_first_year' in teams_df.columns:
                 teams_df['Team_Experience_Years'] = teams_df['year'] - teams_df['team_first_year']
            else:
                 teams_df['Team_Experience_Years'] = 0
        else:
            teams_df['Team_Experience_Years'] = 0

        if drivers_df is not None and not drivers_df.empty:
            drivers_df = drivers_df.merge(
                teams_df[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']],
                on=['Team', 'year'],
                how='left'
            )
            drivers_df['Prev_Year_Team_PTS'] = drivers_df['Prev_Year_Team_PTS'].fillna(0)
            drivers_df['Prev_Year_Team_Pos'] = drivers_df['Prev_Year_Team_Pos'].fillna(default_prev_pos_team)
            drivers_df['Team_Experience_Years'] = drivers_df['Team_Experience_Years'].fillna(0)
        elif drivers_df is not None: 
            drivers_df['Prev_Year_Team_PTS'] = 0
            drivers_df['Prev_Year_Team_Pos'] = default_prev_pos_team # Use the calculated default
            drivers_df['Team_Experience_Years'] = 0
    elif drivers_df is not None: 
            drivers_df['Prev_Year_Team_PTS'] = 0
            drivers_df['Prev_Year_Team_Pos'] = 20 
            drivers_df['Team_Experience_Years'] = 0

    if fastest_laps_df is not None and not fastest_laps_df.empty and \
       'Driver' in fastest_laps_df.columns and 'year' in fastest_laps_df.columns:
        driver_fl_yearly_counts = fastest_laps_df.groupby(['year', 'Driver']).size().reset_index(name='FL_Count_In_Year')
        driver_fl_yearly_counts = driver_fl_yearly_counts.sort_values(by=['Driver', 'year'])
        driver_fl_yearly_counts['Driver_Prev_Season_FL_Count'] = driver_fl_yearly_counts.groupby('Driver')['FL_Count_In_Year'].shift(1).fillna(0)
        
        if drivers_df is not None and not drivers_df.empty:
            drivers_df = drivers_df.merge(
                driver_fl_yearly_counts[['Driver', 'year', 'Driver_Prev_Season_FL_Count']],
                on=['Driver', 'year'],
                how='left'
            )
            drivers_df['Driver_Prev_Season_FL_Count'] = drivers_df['Driver_Prev_Season_FL_Count'].fillna(0)
        elif drivers_df is not None: 
             drivers_df['Driver_Prev_Season_FL_Count'] = 0
    elif drivers_df is not None: 
         drivers_df['Driver_Prev_Season_FL_Count'] = 0

    print("Feature engineering complete.")
    return drivers_df

# ========== 4. DATA RESTRUCTURING FOR MODELING ==========
def restructure_for_modeling(winners_df, drivers_df_with_features, default_prev_pos_driver_val, default_prev_pos_team_val):
    print("Restructuring data for modeling...")
    if winners_df is None or drivers_df_with_features is None:
        print("Error: Missing data for restructuring.")
        return None
    if winners_df.empty:
        print("Error: winners_df is empty for restructuring.")
        return pd.DataFrame() 

    all_samples = []
    if drivers_df_with_features.empty or 'year' not in drivers_df_with_features.columns:
        print("Warning: drivers_df_with_features is empty or missing 'year' for grouping. No samples will be generated.")
        return pd.DataFrame()
        
    drivers_grouped_by_year = {year: group for year, group in drivers_df_with_features.groupby('year')}

    for _, race in winners_df.iterrows():
        current_year = race['year']
        current_gp_name = race['Grand Prix']
        actual_winner_name = race['Winner']
        
        race_country = get_country_from_gp(current_gp_name)

        if current_year not in drivers_grouped_by_year:
            continue 
        year_drivers_df = drivers_grouped_by_year[current_year]

        for _, participant in year_drivers_df.iterrows():
            is_home_race = 0
            participant_nationality = participant.get('Nationality')
            if race_country and isinstance(participant_nationality, str) and race_country == participant_nationality:
                is_home_race = 1
            
            participant_driver_name = participant.get('Driver', 'Unknown Driver From Participant Loop')
            if not isinstance(participant_driver_name, str):
                participant_driver_name = str(participant_driver_name)

            sample = {
                'year': current_year,
                'Grand Prix': current_gp_name,
                'Driver': participant_driver_name,
                'Team': participant.get('Team', 'Unknown Team'),
                'Nationality': participant_nationality if isinstance(participant_nationality, str) else 'Unknown',
                'Prev_Year_Driver_PTS': participant.get('Prev_Year_Driver_PTS', 0),
                'Prev_Year_Driver_Pos': participant.get('Prev_Year_Driver_Pos', default_prev_pos_driver_val),
                'Driver_Experience_Years': participant.get('Driver_Experience_Years', 0),
                'Prev_Year_Team_PTS': participant.get('Prev_Year_Team_PTS', 0),
                'Prev_Year_Team_Pos': participant.get('Prev_Year_Team_Pos', default_prev_pos_team_val),
                'Team_Experience_Years': participant.get('Team_Experience_Years', 0),
                'Driver_Prev_Season_FL_Count': participant.get('Driver_Prev_Season_FL_Count', 0),
                'Is_Home_Race': is_home_race,
                'is_winner': 1 if participant_driver_name == actual_winner_name else 0
            }
            all_samples.append(sample)
    
    if not all_samples: 
        print("Warning: No samples were generated in restructure_for_modeling.")
        return pd.DataFrame()
        
    modeling_df = pd.DataFrame(all_samples)
    print("Data restructuring for modeling complete.")
    return modeling_df

# ========== Global list definitions for columns ==========
CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
NUM_COLS = [
    'year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
    'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
    'Driver_Prev_Season_FL_Count', 'Is_Home_Race'
]
TARGET_COL = 'is_winner'

# ========== 5. PREPROCESSING (ENCODING & SCALING) ==========
def preprocess_data(train_df, test_df, cat_cols, num_cols):
    print("Preprocessing data (encoding and scaling)...")
    label_encoders = {}
    cat_feat_dims = {} 

    # Ensure cat_cols only contains columns present in train_df
    active_cat_cols = [c for c in cat_cols if c in train_df.columns]

    for col in active_cat_cols:
        le = LabelEncoder()
        train_df[col] = train_df[col].astype(str)
        if col in test_df.columns: # Check if column exists in test_df as well
            test_df[col] = test_df[col].astype(str)
        else: # If categorical column is missing in test_df, create it and fill with a placeholder
            print(f"Warning: Categorical column '{col}' not found in test_df. Creating and filling with 'Unknown'.")
            test_df[col] = 'Unknown'

        le.fit(train_df[col])
        train_df[col] = le.transform(train_df[col])
        
        known_classes = set(le.classes_)
        unknown_category_index = len(le.classes_)
        
        test_df[col] = test_df[col].apply(
            lambda x: le.transform([x])[0] if x in known_classes else unknown_category_index
        )
        label_encoders[col] = le
        cat_feat_dims[col] = len(le.classes_) + 1 

    scaler = StandardScaler()
    active_num_cols = [n for n in num_cols if n in train_df.columns] # Ensure num_cols exist in train_df

    if active_num_cols: 
        train_df[active_num_cols] = scaler.fit_transform(train_df[active_num_cols])
        # For test_df, ensure all active_num_cols exist before transform
        missing_num_in_test = [n for n in active_num_cols if n not in test_df.columns]
        if missing_num_in_test:
            print(f"Warning: Numerical columns {missing_num_in_test} not in test_df. Filling with 0 before scaling.")
            for mn_col in missing_num_in_test:
                test_df[mn_col] = 0 # Or median from train, but 0 is simpler for now
        
        # Only transform columns that exist in test_df and were in active_num_cols
        transform_num_cols_test = [n for n in active_num_cols if n in test_df.columns]
        if transform_num_cols_test:
             test_df[transform_num_cols_test] = scaler.transform(test_df[transform_num_cols_test])
    
    print("Preprocessing complete.")
    return train_df, test_df, label_encoders, scaler, cat_feat_dims, active_cat_cols, active_num_cols


# ========== 6. DATASET & DATALOADER ==========
class F1Dataset(Dataset):
    def __init__(self, df_data, cat_cols_ordered, num_cols_ordered, target_col):
        self.actual_cat_cols = [c for c in cat_cols_ordered if c in df_data.columns]
        self.actual_num_cols = [n for n in num_cols_ordered if n in df_data.columns]

        self.cat = df_data[self.actual_cat_cols].values if self.actual_cat_cols else np.empty((len(df_data), 0), dtype=np.long)
        self.num = df_data[self.actual_num_cols].values if self.actual_num_cols else np.empty((len(df_data), 0), dtype=np.float32)
        
        self.y = df_data[target_col].values
        self.num_categorical_feats = self.cat.shape[1]
        self.num_numerical_feats = self.num.shape[1]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        if self.num_categorical_feats > 0:
            cat_features = self.cat[idx]
        else:
            cat_features = np.array([], dtype=np.long) 

        if self.num_numerical_feats > 0:
            num_features = self.num[idx]
        else:
            num_features = np.array([], dtype=np.float32)

        return torch.tensor(cat_features, dtype=torch.long), \
               torch.tensor(num_features, dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.float32)

# ========== 7. MODEL DEFINITION ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims_ordered_list, num_numerical_features, emb_dim=32, hidden_dim=128, dropout_rate=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList()
        total_emb_output_dim = 0
        if cat_dims_ordered_list: 
            for num_unique_values in cat_dims_ordered_list:
                self.embeddings.append(nn.Embedding(num_unique_values, emb_dim))
                total_emb_output_dim += emb_dim
        
        self.num_numerical_features = num_numerical_features
        if self.num_numerical_features > 0:
            self.bn_num = nn.BatchNorm1d(num_numerical_features)
        
        self.fc1_input_dim = total_emb_output_dim + num_numerical_features
        if self.fc1_input_dim == 0: # Should not happen if 'year' is always present
            raise ValueError("Model has no input features (categorical or numerical) for fc1.")

        self.fc1 = nn.Linear(self.fc1_input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1) 

    def forward(self, x_cat, x_num):
        current_features = []
        if self.embeddings: 
            if x_cat.shape[1] != len(self.embeddings): # Check if x_cat has expected number of features
                raise ValueError(f"x_cat has {x_cat.shape[1]} features, but model has {len(self.embeddings)} embedding layers.")
            x_emb_list = []
            for i, emb_layer in enumerate(self.embeddings):
                x_emb_list.append(emb_layer(x_cat[:, i]))
            x_emb = torch.cat(x_emb_list, dim=1)
            current_features.append(x_emb)
        
        if self.num_numerical_features > 0:
            if x_num.shape[1] != self.num_numerical_features: # Check if x_num has expected number of features
                raise ValueError(f"x_num has {x_num.shape[1]} features, but model expects {self.num_numerical_features}.")

            if x_num.ndim == 1 and self.num_numerical_features == 1 : x_num = x_num.unsqueeze(1)
            # No need for the other ndim check, shape check above is better
            
            if x_num.shape[1] > 0 : 
                 x_num_processed = self.bn_num(x_num)
                 current_features.append(x_num_processed)
        
        if not current_features:
             raise ValueError("Model received no input features to concatenate.")

        x = torch.cat(current_features, dim=1)
        
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# ========== 8. TRAINING FUNCTION (remains largely the same) ==========
def train_model(model, trainloader, testloader, n_epoch=30, lr=5e-4, patience=7, model_path='./data/f1_model_final.pth'):
    print(f"Training on {DEVICE}...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    best_test_loss = np.inf
    no_improve_epochs = 0
    train_losses, test_losses = [], []

    for epoch in range(n_epoch):
        model.train()
        epoch_train_loss = 0
        if len(trainloader) == 0:
            print("Warning: Trainloader is empty. Skipping training for this epoch.")
            train_losses.append(float('nan')) 
            if testloader is None or len(testloader) == 0 : 
                test_losses.append(float('nan'))
                continue 
            else: 
                pass 

        for x_cat, x_num, y_true in trainloader:
            x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x_cat, x_num)
            loss = criterion(logits, y_true.unsqueeze(1))
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()
        avg_train_loss = epoch_train_loss / len(trainloader) if len(trainloader) > 0 else 0.0
        train_losses.append(avg_train_loss)

        model.eval()
        epoch_test_loss = 0
        current_loss_for_early_stopping = best_test_loss # Default if no testloader

        if testloader is None or len(testloader) == 0:
            print("Warning: Testloader is empty. Cannot compute test loss for this epoch. Using train loss for early stopping check.")
            test_losses.append(avg_train_loss) 
            current_loss_for_early_stopping = avg_train_loss 
        else:
            with torch.no_grad():
                for x_cat, x_num, y_true in testloader:
                    x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
                    logits = model(x_cat, x_num)
                    loss = criterion(logits, y_true.unsqueeze(1))
                    epoch_test_loss += loss.item()
            avg_test_loss = epoch_test_loss / len(testloader) if len(testloader) > 0 else float('inf')
            test_losses.append(avg_test_loss)
            current_loss_for_early_stopping = avg_test_loss

        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_train_loss:.4f} | Test Loss: {test_losses[-1]:.4f}")

        if current_loss_for_early_stopping < best_test_loss:
            best_test_loss = current_loss_for_early_stopping
            torch.save(model.state_dict(), model_path)
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping triggered after {patience} epochs of no improvement.")
                break
    
    if train_losses and test_losses:
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(test_losses, label='Test Loss')
        plt.xlabel('Epoch')
        plt.ylabel('BCEWithLogitsLoss')
        plt.title('Training & Test Loss Curve')
        plt.legend()
        plt.grid(True)
        plt.savefig('./data/loss_curve_final.png')
        plt.close()
        print("Loss curve saved.")
    return train_losses, test_losses

# ========== 9. EVALUATION FUNCTION (remains largely the same) ==========
def evaluate_model(model, dataloader, model_path_to_load=None):
    if dataloader is None or (hasattr(dataloader, 'dataset') and len(dataloader.dataset) == 0): # Check for empty dataset
        print("Evaluation dataloader is empty or dataset is empty. Skipping evaluation.")
        return

    if model_path_to_load and os.path.exists(model_path_to_load):
        try:
            model.load_state_dict(torch.load(model_path_to_load, map_location=DEVICE))
            print(f"Loaded model from {model_path_to_load} for evaluation.")
        except Exception as e:
            print(f"Error loading model from {model_path_to_load}: {e}. Evaluating with current model state.")
    elif model_path_to_load: 
        print(f"Warning: Model path {model_path_to_load} not found. Evaluating with current model state.")

    model.eval()
    all_y_true, all_y_pred_classes = [], []
    
    with torch.no_grad():
        for x_cat, x_num, y_true in dataloader:
            x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
            logits = model(x_cat, x_num)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).squeeze().int() 
            
            all_y_true.extend(y_true.cpu().numpy())
            if preds.ndim == 0: 
                all_y_pred_classes.append(preds.item())
            else:
                all_y_pred_classes.extend(preds.cpu().numpy())

    if not all_y_true: 
        print("No data to evaluate after processing batches.")
        return

    acc = accuracy_score(all_y_true, all_y_pred_classes)
    all_y_pred_classes_np = np.array(all_y_pred_classes)
    all_y_true_np = np.array(all_y_true)

    report = classification_report(all_y_true_np, all_y_pred_classes_np, target_names=['Not Winner (0)', 'Winner (1)'], zero_division=0)
    cm = confusion_matrix(all_y_true_np, all_y_pred_classes_np)

    print(f"\nEvaluation Results:")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix (Rows: True, Cols: Predicted):")
    print(cm)
    
# ========== 10. GRADIO SETUP & PREDICTION LOGIC ==========
GRADIO_LABEL_ENCODERS = {}
GRADIO_SCALER = None
GRADIO_MODEL = None
GRADIO_DRIVER_INFO_BY_YEAR = {} 
GRADIO_ALL_YEARS = []
GRADIO_ALL_GPS = []
GRADIO_CAT_COLS_ORDERED = []
GRADIO_NUM_COLS_ORDERED = []
default_prev_pos_driver_gradio = 50 
default_prev_pos_team_gradio = 20

def gradio_predict_winner_probabilities(year_input, grand_prix_input):
    global GRADIO_MODEL, GRADIO_LABEL_ENCODERS, GRADIO_SCALER, GRADIO_DRIVER_INFO_BY_YEAR
    global GRADIO_CAT_COLS_ORDERED, GRADIO_NUM_COLS_ORDERED
    global default_prev_pos_driver_gradio, default_prev_pos_team_gradio
    
    scaler_needed_and_missing = bool(GRADIO_NUM_COLS_ORDERED) and (GRADIO_SCALER is None)

    if GRADIO_MODEL is None or not GRADIO_LABEL_ENCODERS or scaler_needed_and_missing :
        return "Error: Model or preprocessors not loaded. Please train/load a model first."

    try:
        year = int(year_input)
    except ValueError:
        return "Error: Invalid year input."
    if not grand_prix_input: # Check if grand_prix_input is empty or None
        return "Error: Grand Prix input cannot be empty."


    if year not in GRADIO_DRIVER_INFO_BY_YEAR or not GRADIO_DRIVER_INFO_BY_YEAR[year]:
        return f"No driver data available for the year {year}."

    participants_for_year = GRADIO_DRIVER_INFO_BY_YEAR[year]
    all_results = []
    GRADIO_MODEL.eval()

    for p_info in participants_for_year: 
        cat_input_values_ordered = []
        if GRADIO_CAT_COLS_ORDERED:
            for col_name in GRADIO_CAT_COLS_ORDERED:
                le = GRADIO_LABEL_ENCODERS[col_name]
                val_to_encode = None
                if col_name == 'Grand Prix': val_to_encode = grand_prix_input
                elif col_name == 'Driver': val_to_encode = p_info.get('Driver', 'Unknown')
                elif col_name == 'Team': val_to_encode = p_info.get('Team', 'Unknown')
                elif col_name == 'Nationality': val_to_encode = p_info.get('Nationality', 'Unknown')
                
                val_to_encode = str(val_to_encode) 
                if val_to_encode in set(le.classes_):
                    cat_input_values_ordered.append(le.transform([val_to_encode])[0])
                else:
                    cat_input_values_ordered.append(len(le.classes_)) 
            x_cat_tensor = torch.tensor([cat_input_values_ordered], dtype=torch.long).to(DEVICE)
        else: 
            x_cat_tensor = torch.empty(1, 0, dtype=torch.long).to(DEVICE)
        
        num_input_values_ordered = []
        if GRADIO_NUM_COLS_ORDERED:
            for col_name in GRADIO_NUM_COLS_ORDERED:
                if col_name == 'year': val = float(year)
                elif col_name == 'Prev_Year_Driver_PTS': val = p_info.get('Prev_Year_Driver_PTS', 0)
                elif col_name == 'Prev_Year_Driver_Pos': val = p_info.get('Prev_Year_Driver_Pos', default_prev_pos_driver_gradio) 
                elif col_name == 'Driver_Experience_Years': val = p_info.get('Driver_Experience_Years', 0)
                elif col_name == 'Prev_Year_Team_PTS': val = p_info.get('Prev_Year_Team_PTS', 0)
                elif col_name == 'Prev_Year_Team_Pos': val = p_info.get('Prev_Year_Team_Pos', default_prev_pos_team_gradio) 
                elif col_name == 'Team_Experience_Years': val = p_info.get('Team_Experience_Years', 0)
                elif col_name == 'Driver_Prev_Season_FL_Count': val = p_info.get('Driver_Prev_Season_FL_Count', 0)
                elif col_name == 'Is_Home_Race':
                    race_country_gradio = get_country_from_gp(grand_prix_input)
                    is_home = 0
                    participant_nat = p_info.get('Nationality')
                    if race_country_gradio and isinstance(participant_nat, str) and race_country_gradio == participant_nat:
                        is_home = 1
                    val = float(is_home)
                else: 
                    val = 0 
                num_input_values_ordered.append(float(val))
            
            x_num_np = np.array([num_input_values_ordered], dtype=np.float32)
            # *** UserWarning FIX START ***
            x_num_df_for_transform = pd.DataFrame(x_num_np, columns=GRADIO_NUM_COLS_ORDERED)
            x_num_scaled = GRADIO_SCALER.transform(x_num_df_for_transform)
            # *** UserWarning FIX END ***
            x_num_tensor = torch.tensor(x_num_scaled, dtype=torch.float32).to(DEVICE)
        else: 
            x_num_tensor = torch.empty(x_cat_tensor.shape[0], 0, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            logits = GRADIO_MODEL(x_cat_tensor, x_num_tensor)
            probability = torch.sigmoid(logits).cpu().item()
        
        all_results.append((p_info.get('Driver', 'N/A'), p_info.get('Team', 'N/A'), probability))

    all_results.sort(key=lambda x: x[2], reverse=True)
    output_text = f"Predictions for {grand_prix_input}, {year}:\n"
    for i, (driver, team, prob) in enumerate(all_results[:10], 1): 
        output_text += f"{i}. {driver} ({team}): {prob:.2%}\n"
    return output_text

# ========== 11. MAIN EXECUTION ==========
if __name__ == '__main__':
    default_prev_pos_driver_main = 50 
    default_prev_pos_team_main = 20

    winners_df, drivers_raw_df, teams_raw_df, fastest_laps_raw_df = load_and_clean_data()

    if any(df is None for df in [winners_df, drivers_raw_df, teams_raw_df, fastest_laps_raw_df]):
        print("Exiting due to data loading errors.")
        exit()
    
    if drivers_raw_df is not None and 'Pos' in drivers_raw_df.columns and not drivers_raw_df['Pos'].dropna().empty:
        max_pos_val_d = drivers_raw_df['Pos'].max(skipna=True)
        # Ensure max_pos_val_d is not NaN before int conversion
        default_prev_pos_driver_gradio = (int(max_pos_val_d) + 5) if pd.notna(max_pos_val_d) and max_pos_val_d > 0 else 50
        default_prev_pos_driver_main = default_prev_pos_driver_gradio
        
    if teams_raw_df is not None and 'Pos' in teams_raw_df.columns and not teams_raw_df['Pos'].dropna().empty:
        max_pos_val_t = teams_raw_df['Pos'].max(skipna=True)
        default_prev_pos_team_gradio = (int(max_pos_val_t) + 5) if pd.notna(max_pos_val_t) and max_pos_val_t > 0 else 20
        default_prev_pos_team_main = default_prev_pos_team_gradio

    drivers_featured_df = engineer_features(winners_df, drivers_raw_df, teams_raw_df, fastest_laps_raw_df)
    
    if drivers_featured_df is None or drivers_featured_df.empty:
        print("Exiting as drivers_featured_df is empty or None after feature engineering.")
        exit()

    modeling_df = restructure_for_modeling(winners_df, drivers_featured_df, default_prev_pos_driver_main, default_prev_pos_team_main)

    if modeling_df is None or modeling_df.empty:
        print("Exiting as modeling_df is empty or None.")
        exit()
    
    modeling_df['year_str'] = modeling_df['year'].astype(str)
    modeling_df['gp_str'] = modeling_df['Grand Prix'].astype(str)
    modeling_df['race_id'] = modeling_df['year_str'] + "_" + modeling_df['gp_str']
    
    unique_race_ids = modeling_df['race_id'].unique()
    if len(unique_race_ids) < 2 :
        print("Error: Not enough unique races to perform train-test split. Need at least 2.")
        train_df = modeling_df.copy()
        test_df = modeling_df.copy()
        if not modeling_df.empty: print("Warning: Using all data for both training and testing due to insufficient unique races for a split.")
        else: exit()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED, shuffle=True)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    if train_df.empty:
        print("Error: Train DataFrame is empty after split. Check data and split logic.")
        exit()

    for df_to_clean in [train_df, test_df]:
        if not df_to_clean.empty:
             cols_to_drop = ['year_str', 'gp_str', 'race_id']
             existing_cols_to_drop = [c for c in cols_to_drop if c in df_to_clean.columns]
             if existing_cols_to_drop:
                 df_to_clean.drop(columns=existing_cols_to_drop, inplace=True)

    print("\nNaN check and imputation in training/testing features (before encoding/scaling):")
    for col in CAT_COLS + NUM_COLS: 
        if col in train_df.columns: 
            if train_df[col].isnull().any():
                print(f"Column '{col}' in train_df has NaNs: {train_df[col].isnull().sum()} occurrences. Imputing...")
                if train_df[col].dtype == 'object' or \
                   pd.api.types.is_categorical_dtype(train_df[col]) or \
                   pd.api.types.is_string_dtype(train_df[col]):
                    train_df[col] = train_df[col].fillna('Unknown') 
                    if col in test_df.columns: test_df[col] = test_df[col].fillna('Unknown')
                else: 
                    median_val = train_df[col].median()
                    train_df[col] = train_df[col].fillna(median_val)
                    if col in test_df.columns: test_df[col] = test_df[col].fillna(median_val)
        # Create column if it's defined in CAT_COLS/NUM_COLS but missing (e.g., if a feature was all NaN and dropped)
        # This ensures preprocess_data receives all expected columns.
        for df_set in [train_df, test_df]:
             if col not in df_set.columns:
                 print(f"Warning: Column '{col}' was expected but not found in DataFrame. Adding it with default values.")
                 if col in CAT_COLS : df_set[col] = 'Unknown'
                 else: df_set[col] = 0 # Default for missing numerical columns


    train_df, test_df, label_encoders_map, scaler_obj, cat_feat_dims_map, \
        final_cat_cols, final_num_cols = preprocess_data(train_df, test_df, CAT_COLS, NUM_COLS)

    train_dataset = F1Dataset(train_df, final_cat_cols, final_num_cols, TARGET_COL)
    test_dataset = F1Dataset(test_df, final_cat_cols, final_num_cols, TARGET_COL) if not test_df.empty else None

    if len(train_dataset) == 0:
        print("Error: Training dataset is empty. Cannot proceed.")
        exit()

    class_counts_train = train_df[TARGET_COL].value_counts().to_dict()
    weight_class_0 = 1. / (class_counts_train.get(0, 1) + 1e-6) 
    weight_class_1 = 1. / (class_counts_train.get(1, 1) + 1e-6)
    samples_weight_train = np.array([weight_class_1 if t == 1 else weight_class_0 for t in train_df[TARGET_COL]])
    sampler_train = WeightedRandomSampler(torch.from_numpy(samples_weight_train).double(), len(samples_weight_train))

    train_loader = DataLoader(train_dataset, batch_size=256, sampler=sampler_train)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False) if test_dataset and len(test_dataset) > 0 else None

    ordered_cat_dims_list = [cat_feat_dims_map[col] for col in final_cat_cols if col in cat_feat_dims_map]
    num_numerical_features_model = len(final_num_cols)

    f1_model = F1DNN(ordered_cat_dims_list, num_numerical_features_model, emb_dim=32, hidden_dim=128, dropout_rate=0.4).to(DEVICE)
    
    MODEL_PATH = './data/f1_model_final_v2.pth'
    TRAIN_MODEL_FLAG = not os.path.exists(MODEL_PATH) 

    if TRAIN_MODEL_FLAG:
        print(f"Starting model training as TRAIN_MODEL_FLAG is True or model path '{MODEL_PATH}' not found.")
        effective_test_loader = test_loader if test_loader and len(test_loader) > 0 else train_loader 
        if len(train_loader) == 0:
             print("Error: Train loader is empty. Skipping training.")
        else:
             train_model(f1_model, train_loader, effective_test_loader, n_epoch=50, lr=5e-4, patience=10, model_path=MODEL_PATH)
    else:
        print(f"Skipping training. Loading model from {MODEL_PATH}")

    print("\n--- Final Evaluation on Test Set ---")
    # Ensure model is loaded if not trained in this session but file exists
    if not TRAIN_MODEL_FLAG and os.path.exists(MODEL_PATH):
        try:
            f1_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
            print(f"Successfully loaded model from {MODEL_PATH} for evaluation.")
        except Exception as e:
            print(f"Error loading model from {MODEL_PATH} for final evaluation: {e}")


    if test_loader and len(test_loader) > 0 :
        evaluate_model(f1_model, test_loader) # Removed model_path_to_load, as it's handled above or model is already trained
    else:
        print("Test loader is empty or None, skipping final evaluation on test set. Evaluating on training data instead (for diagnostics).")
        evaluate_model(f1_model, train_loader)


    GRADIO_MODEL = f1_model
    GRADIO_LABEL_ENCODERS = label_encoders_map
    GRADIO_SCALER = scaler_obj
    GRADIO_CAT_COLS_ORDERED = final_cat_cols 
    GRADIO_NUM_COLS_ORDERED = final_num_cols
    
    if drivers_featured_df is not None and not drivers_featured_df.empty:
        temp_drivers_info_for_gradio = drivers_featured_df.copy()
        for year_val, group in temp_drivers_info_for_gradio.groupby('year'):
            GRADIO_DRIVER_INFO_BY_YEAR[year_val] = group.to_dict('records')
        if 'year' in drivers_featured_df.columns:
            GRADIO_ALL_YEARS = sorted(list(drivers_featured_df['year'].unique()))
        else: GRADIO_ALL_YEARS = [2023] 
    else: 
        GRADIO_ALL_YEARS = [2023] 
    
    if winners_df is not None and not winners_df.empty and 'Grand Prix' in winners_df.columns:
         GRADIO_ALL_GPS = sorted(list(winners_df['Grand Prix'].unique()))
    else: 
         GRADIO_ALL_GPS = ["Monaco Grand Prix"]


    print("\nLaunching Gradio Interface...")
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# F1 Grand Prix Winner Probability Predictor")
        gr.Markdown("Predicts the win probability for each participating driver in a selected F1 race.")
        
        with gr.Row():
            year_dropdown_gradio = gr.Dropdown(label="Select Year", choices=GRADIO_ALL_YEARS, value=GRADIO_ALL_YEARS[-1] if GRADIO_ALL_YEARS else None)
            gp_dropdown_gradio = gr.Dropdown(label="Select Grand Prix", choices=GRADIO_ALL_GPS, value=GRADIO_ALL_GPS[0] if GRADIO_ALL_GPS else None)
        
        predict_button_gradio = gr.Button("Predict Probabilities", variant="primary")
        output_textbox_gradio = gr.Textbox(label="Predicted Win Probabilities (Top 10)", lines=12, interactive=False)
        
        predict_button_gradio.click(
            gradio_predict_winner_probabilities,
            inputs=[year_dropdown_gradio, gp_dropdown_gradio],
            outputs=[output_textbox_gradio]
        )
        
        with gr.Accordion("Training Information", open=False):
            gr.Markdown("Model training loss curve:")
            loss_curve_path_gradio = "./data/loss_curve_final.png"
            if os.path.exists(loss_curve_path_gradio):
                gr.Image(value=loss_curve_path_gradio, label="Loss Curve")
            else:
                gr.Markdown(f"Loss curve image ({loss_curve_path_gradio}) not found. Train the model to generate it.")

    demo.launch(share=False)

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and cleaning data...
Excluding Indianapolis 500 races...
Data loading and initial cleaning complete.
Engineering features...
Feature engineering complete.
Restructuring data for modeling...
Data restructuring for modeling complete.

NaN check in training features (before imputation/encoding):
Column 'Team' in train_df has NaNs: 80 occurrences.
Preprocessing data (encoding and scaling)...
Preprocessing complete.
Skipping training. Loading model from ./data/f1_model_final_v2.pth

--- Final Evaluation on Test Set ---
Loaded model from ./data/f1_model_final_v2.pth for evaluation.

Evaluation Results:
Accuracy: 0.8574
Classification Report:
                precision    recall  f1-score   support

Not Winner (0)       0.98      0.86      0.92      4710
    Winner (1)       0.19      0.70      0.30       220

      accuracy                           0.86      4930
     macro avg       0.59      0.78      0.61      4930
  weighted avg       0.95      0.86      0.89      4930

Confusion

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f